In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'presentation'
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


c:\Users\ahmad\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\ahmad\anaconda3\lib\site-packages\numpy\.libs\libopenblas.EL2C6PLE4ZYW3ECEVIV3OXXGRN2NRFM2.gfortran-win_amd64.dll
c:\Users\ahmad\anaconda3\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


# Dataset

In [2]:
df = pd.DataFrame({'x1': [2, 3, 4, -2, -4, -3], 'x2': [1, 5, 3, -2, -3, -5]})
df

,x1,x2
0,2,1
1,3,5
2,4,3
3,-2,-2
4,-4,-3
5,-3,-5


In [3]:
fig = px.scatter(df, x='x1', y='x2', width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.show()


# PCA Model from Scratch


In [4]:
# Standardize the data

scaler = StandardScaler()
df_standardized = scaler.fit_transform(df)
df_standardized

array([[ 0.64326752,  0.33485541],
       [ 0.96490128,  1.48293111],
       [ 1.28653504,  0.90889326],
       [-0.64326752, -0.52620136],
       [-1.28653504, -0.81322028],
       [-0.96490128, -1.38725813]])

In [5]:
fig = px.scatter(x=df_standardized[:, 0], y=df_standardized[:, 1], width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.show()

In [6]:
# calculate covariance matrix

covariance_matrix = np.cov(df_standardized.T)
covariance_matrix

array([[1.2       , 1.10777971],
       [1.10777971, 1.2       ]])

In [7]:
# calculate eigenvalues and eigenvectors

eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)
print('Eigenvalues:\n', eigenvalues)
print('Eigenvectors:\n', eigenvectors)

Eigenvalues:
 [0.09222029 2.30777971]
Eigenvectors:
 [[-0.70710678 -0.70710678]
 [ 0.70710678 -0.70710678]]


In [8]:
idx = eigenvalues.argsort()
idx[::-1]

array([1, 0], dtype=int64)

In [9]:
# sort eigenvalues and eigenvectors

idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print('Eigenvalues:\n', eigenvalues)
print('Eigenvectors:\n', eigenvectors)

Eigenvalues:
 [2.30777971 0.09222029]
Eigenvectors:
 [[-0.70710678 -0.70710678]
 [-0.70710678  0.70710678]]


In [10]:
# calculate explained variance ratio

explained_variance_ratio = eigenvalues / sum(eigenvalues)
explained_variance_ratio

array([0.96157488, 0.03842512])

In [11]:
# calculate singular values

singular_values = np.sqrt(eigenvalues)
singular_values


array([1.51913782, 0.30367794])

In [12]:
np.__version__

'1.23.5'

In [13]:
#update numpy
#!pip install numpy --upgrade

In [14]:
# transform the data

df_transformed = np.dot(df_standardized, eigenvectors)
df_transformed

array([[-0.69163736, -0.21808029],
       [-1.73087888,  0.3663024 ],
       [-1.55240224, -0.26703307],
       [ 0.82693938,  0.08277828],
       [ 1.48475123,  0.33468407],
       [ 1.66322787, -0.29865139]])

# Compare with PCA from Sklearn


In [15]:
pca = PCA(n_components=2)
pca.fit(df_standardized)
print('PCA components:\n', pca.components_)
print('PCA explained variance ratio:\n', pca.explained_variance_ratio_)

PCA components:
 [[ 0.70710678  0.70710678]
 [-0.70710678  0.70710678]]
PCA explained variance ratio:
 [0.96157488 0.03842512]


In [16]:
# check 2 matrices are equal

np.allclose(pca.components_.T, eigenvectors)

False

In [17]:
# transform the data
df_transformed = pca.transform(df_standardized)
df_transformed

array([[ 0.69163736, -0.21808029],
       [ 1.73087888,  0.3663024 ],
       [ 1.55240224, -0.26703307],
       [-0.82693938,  0.08277828],
       [-1.48475123,  0.33468407],
       [-1.66322787, -0.29865139]])

In [18]:
# PLot the transformed data
df_transformed = pd.DataFrame(df_transformed, columns=['PC1', 'PC2'])

fig = px.scatter(df_transformed, x='PC1', y='PC2', width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.show()

In [19]:
# Plot the pc components as vectors

fig = px.scatter(df, x='x1', y='x2', width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0], y1=eigenvectors[1, 0], line=dict(color='red', width=3))
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 1], y1=eigenvectors[1, 1], line=dict(color='red', width=3))
fig.show()

In [20]:
# Plot the pc components on the original data

fig = px.scatter(df, x='x1', y='x2', width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0]*10, y1=eigenvectors[1, 0]*10, line=dict(color='red', width=3))
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 1]*10, y1=eigenvectors[1, 1]*10, line=dict(color='red', width=3))
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0]*-10, y1=eigenvectors[1, 0]*-10, line=dict(color='red', width=3))
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 1]*-10, y1=eigenvectors[1, 1]*-10, line=dict(color='red', width=3))
fig.show()


In [21]:
# Measure the angle between the pc components
angle = np.arccos(np.dot(eigenvectors[:, 0], eigenvectors[:, 1]) / (np.linalg.norm(eigenvectors[:, 0]) * np.linalg.norm(eigenvectors[:, 1])))

# angle in degrees
angle * 180 / np.pi

90.0

In [22]:
# Project the data onto the first principal component
df_projected = np.dot(df, eigenvectors[:, 0])

# inverse transform the projected data
df_reconstructed = np.dot(df_projected.reshape(-1, 1), eigenvectors[:, 0].reshape(1, -1))
df_reconstructed

array([[ 1.5,  1.5],
       [ 4. ,  4. ],
       [ 3.5,  3.5],
       [-2. , -2. ],
       [-3.5, -3.5],
       [-4. , -4. ]])

In [23]:
# plot the original data and the projected data on the first principal component
fig = px.scatter(df, x='x1', y='x2', width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0]*10, y1=eigenvectors[1, 0]*10, line=dict(color='red', width=3))
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0]*-10, y1=eigenvectors[1, 0]*-10, line=dict(color='red', width=3))
fig.add_trace(go.Scatter(x=df_reconstructed[:, 0], y=df_reconstructed[:, 1], mode='markers', marker=dict(size=10, color='red', line=dict(width=2, color='black'))))
fig.show()


In [24]:
# Plot PC1
fig = px.scatter(df_reconstructed, x=0, y=1, width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0]*10, y1=eigenvectors[1, 0]*10, line=dict(color='red', width=3))
fig.add_shape(type='line', x0=0, y0=0, x1=eigenvectors[0, 0]*-10, y1=eigenvectors[1, 0]*-10, line=dict(color='red', width=3))
fig.show()

In [25]:
df_reconstructed[0, :]

array([1.5, 1.5])

In [26]:
df_reconstructed[0, :].dot(eigenvectors[:, 0])

-2.1213203435596424

In [27]:
df_reconstructed[0, :].dot(eigenvectors[:, 1])

0.0

In [28]:
np.sqrt(1.5**2 + 1.5**2)

2.1213203435596424